# Recurrent Neural Networks in TensorFlow/Keras

This notebook builds an LSTM-based sentiment classifier on the IMDB movie review dataset.


## Learning objectives
- Work with padded sequence data.
- Build an embedding + LSTM model in Keras.
- Train and evaluate a text classification model.
- Explore predictions on new review snippets.


In [ ]:
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)


## Load the IMDB dataset
The IMDB dataset contains integer-encoded movie reviews labeled as positive or negative.


In [ ]:
vocab_size = 10000
maxlen = 200

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
len(X_train), len(X_test)


## Pad the sequences
Padding ensures that every review has the same sequence length within a batch.


In [ ]:
X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test, maxlen=maxlen)

X_train.shape, X_test.shape


## Reserve a validation set
We split off 5,000 reviews from the training set for validation.


In [ ]:
X_valid = X_train[:5000]
X_partial = X_train[5000:]
y_valid = y_train[:5000]
y_partial = y_train[5000:]

X_partial.shape, X_valid.shape


## Build an LSTM classifier
The embedding layer learns dense word vectors before the LSTM processes the sequence.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 128, input_length=maxlen),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.summary()


## Compile the model
Binary crossentropy matches the two-class sentiment prediction task.


In [ ]:
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])


## Train the network
In practice, you may add `EarlyStopping` to reduce overfitting.


In [ ]:
history = model.fit(
    X_partial, y_partial,
    epochs=4,
    batch_size=64,
    validation_data=(X_valid, y_valid),
    verbose=2
)


## Evaluate on the test set
This gives the final held-out sentiment classification performance.


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


## Predict review sentiment
Scores close to 1 indicate positive sentiment; scores near 0 indicate negative sentiment.


In [ ]:
pred_scores = model.predict(X_test[:5]).flatten()
print("Scores:", np.round(pred_scores, 4))
print("Predicted labels:", (pred_scores >= 0.5).astype(int))
print("Actual labels:   ", y_test[:5])


## Optional extension
Replace the bidirectional LSTM with a GRU, stack recurrent layers, or compare performance with a 1D CNN for text classification.
